# Geotagger Evaluation

Calls `/geotag` for each fixture and measures:
- **City precision** — `geo_cities[0].city_name` matches `expected_city`
- **Scope accuracy** — `geo_scope` matches `expected_geo_scope`
- **Street detection** — `geo_streets` count vs `expected_streets_min`
- **Place recall** — `all_places` count vs `expected_places_min`

The geotagger now runs in two explicit stages:
- **Stage A** — toponym identification: PlanTL RoBERTa NER + street-prefix regex  
- **Stage B** — resolution: city-first (B1), street index lookup (B2), region/points (B3)

Response fields: `geo_scope`, `geo_region`, `geo_cities[]`, `geo_streets[]`, `geo_points[]`, `all_places[]`  
Legacy fields `city` and `city_confidence` are preserved for backward compat.

**Prerequisite:** NLP service running at `http://localhost:8001` (`/readyz` returns 200).

In [1]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('geotag_cases.json')
scope_cases = [c for c in cases if c.get('expected_geo_scope')]
street_cases = [c for c in cases if c.get('expected_streets_min', 0) > 0]
print(f'Loaded {len(cases)} test cases  ({len(scope_cases)} with scope expectations, {len(street_cases)} with street expectations)')

Loaded 10 test cases  (10 with scope expectations, 2 with street expectations)


In [2]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, 'Service not ready'

AssertionError: Service not ready

In [5]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/geotag',
        json={
            'article_id': case['article_id'],
            'text': case['text'],
            'headline': case.get('headline', ''),
            'source': case.get('source', ''),
            'scope_signal': case.get('scope_signal_hint'),   # pre-computed signal from classifier (optional)
        },
        headers=HEADERS
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, f"{case['id']}: HTTP {resp.status_code}"
    data = resp.json()

    expected_city      = case.get('expected_city')
    expected_geo_scope = case.get('expected_geo_scope')
    expected_places_min  = case.get('expected_places_min', 0)
    expected_streets_min = case.get('expected_streets_min', 0)

    # City: prefer geo_cities list, fall back to legacy city field
    geo_cities  = data.get('geo_cities', [])
    returned_city = geo_cities[0]['city_name'] if geo_cities else data.get('city')
    returned_conf = geo_cities[0]['confidence'] if geo_cities else data.get('city_confidence', 0)
    city_correct = (
        (expected_city is None and not geo_cities) or
        (expected_city is not None and returned_city == expected_city)
    )

    # Scope
    returned_scope = data.get('geo_scope')
    scope_correct = (expected_geo_scope is None) or (returned_scope == expected_geo_scope)

    # Streets
    geo_streets = data.get('geo_streets', [])
    streets_ok = len(geo_streets) >= expected_streets_min

    places_ok = len(data.get('all_places', [])) >= expected_places_min
    passed = city_correct and scope_correct and streets_ok and places_ok
    icon = '✅' if passed else '❌'

    results.append({
        'id': case['id'],
        'description': case['description'],
        'expected_city': expected_city,
        'returned_city': returned_city,
        'city_confidence': returned_conf,
        'city_correct': city_correct,
        'returned_scope': returned_scope,
        'expected_scope': expected_geo_scope,
        'scope_correct': scope_correct,
        'geo_cities': geo_cities,
        'geo_streets': geo_streets,
        'geo_points': data.get('geo_points', []),
        'geo_region': data.get('geo_region'),
        'streets_ok': streets_ok,
        'places_found': len(data.get('all_places', [])),
        'places_ok': places_ok,
        'latency_s': latency,
        'passed': passed,
    })

    print(f"{icon} [{case['id']}] city={returned_city!r} (expected={expected_city!r})  scope={returned_scope!r}  conf={returned_conf:.2f}  places={len(data.get('all_places',[]))}  {latency:.1f}s")
    if geo_streets:
        print(f"   streets: {[s['span'] for s in geo_streets[:3]]}")
    if data.get('geo_region'):
        print(f"   region:  {data['geo_region']}")
    if data.get('geo_points'):
        print(f"   points:  {[p['span'] for p in data['geo_points'][:3]]}")
    if not city_correct:
        print(f"   ⚠ CITY MISMATCH — {case['description']}")
    if not scope_correct:
        print(f"   ⚠ SCOPE MISMATCH — expected={expected_geo_scope!r} got={returned_scope!r}")
    if not streets_ok:
        print(f"   ⚠ STREET MISS — expected ≥{expected_streets_min} streets, got {len(geo_streets)}")
    print()

❌ [geo-001] city=None (expected='Madrid')  scope=None  conf=0.00  places=2  0.3s
   ⚠ CITY MISMATCH — Single unambiguous city mention
   ⚠ SCOPE MISMATCH — expected='city' got=None

❌ [geo-002] city=None (expected='Valladolid')  scope=None  conf=0.00  places=1  0.2s
   ⚠ CITY MISMATCH — City with accent (common Spanish name)
   ⚠ SCOPE MISMATCH — expected='city' got=None

❌ [geo-003] city='Madrid' (expected='Barcelona')  scope=None  conf=0.58  places=4  0.3s
   ⚠ CITY MISMATCH — Multiple city mentions — should pick the dominant one
   ⚠ SCOPE MISMATCH — expected='city' got=None

❌ [geo-004] city=None (expected='Barcelona')  scope=None  conf=0.00  places=2  0.3s
   ⚠ CITY MISMATCH — Street mention without explicit city — city inferred from DB snapshot
   ⚠ SCOPE MISMATCH — expected='city' got=None
   ⚠ STREET MISS — expected ≥1 streets, got 0

❌ [geo-005] city=None (expected=None)  scope=None  conf=0.00  places=1  0.2s
   ⚠ SCOPE MISMATCH — expected='national' got=None

❌ [geo-006] city

In [4]:
passing = [r for r in results if r['passed']]
city_precision  = sum(1 for r in results if r['city_correct']) / len(results)
scope_accuracy  = sum(1 for r in results if r['scope_correct']) / len(results)
streets_total   = sum(len(r['geo_streets']) for r in results)
street_cases_ok = sum(1 for r in results if r['streets_ok'])
street_cases_n  = sum(1 for c in cases if c.get('expected_streets_min', 0) > 0)
avg_conf = sum(r['city_confidence'] for r in results) / len(results)
avg_lat  = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('GEOTAGGER', {
    'Cases': len(results),
    'Fully passing': f"{len(passing)}/{len(results)}",
    'City precision': city_precision,
    'Scope accuracy': scope_accuracy,
    'Street cases passing': f"{street_cases_ok}/{street_cases_n}",
    'Streets found (total)': streets_total,
    'Target city precision': 0.85,
    'Average city confidence': avg_conf,
    'Average latency (s)': avg_lat,
})

# Scope breakdown
from collections import Counter
scope_counter = Counter(r['returned_scope'] for r in results)
print('Scope distribution:')
for scope, n in sorted(scope_counter.items(), key=lambda x: str(x[0])):
    print(f'  {scope!r:<12} {n} articles')


  GEOTAGGER
  Cases                               10
  Fully passing                       0/10
  City precision                      0.300
  Scope accuracy                      0.000
  Street cases passing                8/2
  Streets found (total)               0
  Target city precision               0.850
  Average city confidence             0.082
  Average latency (s)                 0.251

Scope distribution:
  None         10 articles


## Tuning Guide

### Stage A — Toponym identification (NER + regex)

| Symptom | Lever | Where |
|---------|-------|-------|
| City names not found (missing spans) | Model not capturing entity | `nlp/geotagger/ner.py` — check `_KEEP_LABELS` |
| Street spans duplicated by NER and regex | Overlap check incorrect | `ner.py: extract_spans()` — `covered` set logic |
| Street names captured with trailing noise | Tighten regex quantifier | `ner.py: _STREET_RE` — reduce `{2,50}` upper bound |

### Stage B1 — City resolution

| Symptom | Lever | Where |
|---------|-------|-------|
| Wrong city for ambiguous articles | Increase headline weight | `disambiguator.py: score_candidates()` — `in_title` weight |
| City not found despite clear mention | Check GeoNames data | Re-run `build_geonames_es.py`; verify `cities_snapshot.json` has city |
| Low confidence on unambiguous articles | Review scoring formula | `disambiguator.py` — frequency + population + title weights |
| Source prior not helping | Populate prior config | `config/source_city_prior.json` — add `"source_name": city_id` |

### Stage B2 — Street resolution

| Symptom | Lever | Where |
|---------|-------|-------|
| Streets found but `edge_ids=[]` | Index not populated | Run `scripts/snapshot_streets.py` to build `street_index.json` |
| Street matched to wrong city | City resolved incorrectly in B1 | Fix B1 city first |
| Street span not normalized correctly | Check prefix stripping | `gazetteer.py: _normalize_street()` |

### Stage B3 — Scope / region / points

| Symptom | Lever | Where |
|---------|-------|-------|
| `geo_scope` wrong on national articles | No geo evidence → fallback to national | Pass `scope_signal` from classifier |
| Regional articles get `scope=city` | GeoNames A-class entries missing | Re-run GeoNames build |
| City articles get `scope=national` | City not resolved (B1 returned None) | Fix city resolution first |